# 🤖 Model Training & Evaluation
Loads preprocessed data and trains three models in order of complexity.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve
)
from xgboost import XGBClassifier

sns.set_theme(style='darkgrid')
print('Libraries loaded ✅')

## 1. Load Preprocessed Data

In [ ]:
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_tfidf_train    = data['X_tfidf_train']
X_tfidf_test     = data['X_tfidf_test']
X_combined_train = data['X_combined_train']
X_combined_test  = data['X_combined_test']
X_text_train     = data['X_text_train']
X_text_test      = data['X_text_test']
y_train          = data['y_train']
y_test           = data['y_test']

print('Data loaded ✅')
print(f'Train: {len(y_train):,} | Test: {len(y_test):,}')

## 2. Helper: Evaluation Function
Reused for every model to keep results consistent.

In [ ]:
results = {}  # stores all model results for final comparison

def evaluate(name, y_true, y_pred, y_prob):
    metrics = {
        'Accuracy' : accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall'   : recall_score(y_true, y_pred),
        'F1'       : f1_score(y_true, y_pred),
        'ROC-AUC'  : roc_auc_score(y_true, y_prob)
    }
    results[name] = metrics

    print(f'\n--- {name} ---')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fake', 'Real'],
                yticklabels=['Fake', 'Real'])
    plt.title(f'Confusion Matrix — {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    return metrics

---
## 3. Model 1 — Baseline: TF-IDF + Logistic Regression
> Fast, interpretable, strong baseline. Uses TF-IDF vectors only.

In [ ]:
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', n_jobs=-1)
lr.fit(X_tfidf_train, y_train)

lr_pred = lr.predict(X_tfidf_test)
lr_prob = lr.predict_proba(X_tfidf_test)[:, 1]

evaluate('Logistic Regression', y_test, lr_pred, lr_prob)

In [ ]:
# Top predictive words for each class
tfidf_vocab = data['tfidf'].get_feature_names_out()
coefs = lr.coef_[0]

top_fake = pd.Series(coefs, index=tfidf_vocab).nsmallest(20)
top_real = pd.Series(coefs, index=tfidf_vocab).nlargest(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_fake.plot(kind='barh', ax=axes[0], color='#e74c3c')
axes[0].set_title('Top 20 Words → Fake')
top_real.plot(kind='barh', ax=axes[1], color='#2ecc71')
axes[1].set_title('Top 20 Words → Real')
plt.tight_layout()
plt.show()

---
## 4. Model 2 — XGBoost + Engineered Features
> Combines TF-IDF with handcrafted features (word count, caps ratio, etc.) from preprocessing.

In [ ]:
print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)
xgb.fit(X_combined_train, y_train)

xgb_pred = xgb.predict(X_combined_test)
xgb_prob = xgb.predict_proba(X_combined_test)[:, 1]

evaluate('XGBoost', y_test, xgb_pred, xgb_prob)

---
## 5. Model 3 — Fine-tuned DistilBERT
> ⚠️ GPU recommended. Run on your partner's machine or Google Colab.
> Runtime: ~30–45 min on a T4 GPU, much longer on CPU.

In [ ]:
# Install if needed
# !pip install transformers torch datasets

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = NewsDataset(X_text_train, y_train.tolist(), tokenizer)
test_dataset  = NewsDataset(X_text_test,  y_test.tolist(),  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=EPOCHS * len(train_loader)
)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f}')

print('Training complete ✅')

In [ ]:
model.eval()
all_preds, all_probs = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
        probs          = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        preds          = (probs >= 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)

evaluate('DistilBERT', y_test, all_preds, all_probs)

In [ ]:
# Save the model
model.save_pretrained('./distilbert_model')
tokenizer.save_pretrained('./distilbert_model')
print('DistilBERT model saved to ./distilbert_model ✅')

---
## 6. Final Model Comparison

In [ ]:
comparison = pd.DataFrame(results).T.round(4)
print(comparison.to_string())

comparison.plot(kind='bar', figsize=(12, 5), edgecolor='black')
plt.title('Model Comparison — All Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0.7, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all models
plt.figure(figsize=(8, 6))

for name, y_prob, color in zip(
    ['Logistic Regression', 'XGBoost', 'DistilBERT'],
    [lr_prob, xgb_prob, all_probs],
    ['#3498db', '#e67e22', '#9b59b6']
):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

plt.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend()
plt.tight_layout()
plt.show()